# Re-téléchargement des GPX corrompus (502)
Cible uniquement les fichiers GPX qui contiennent une page d'erreur HTML au lieu d'un vrai GPX.

In [7]:
import os
import re
import time
import pandas as pd
import requests
from tqdm import tqdm
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

WORK_DIR    = '/Users/arthurdeletang/Desktop/Stage M1/Code'
GPX_DIR     = os.path.join(WORK_DIR, 'data', 'gpx_files_2')
OUTPUT_BASE = os.path.join(WORK_DIR, 'data', 'matching')
RACES_CSV   = os.path.join(WORK_DIR, 'data', 'races.csv')


In [8]:
# ── 1. Identifier les fichiers GPX corrompus (contenu HTML 502) ─────────────
corrompus = []
for fname in os.listdir(GPX_DIR):
    if not fname.endswith('.gpx'):
        continue
    path = os.path.join(GPX_DIR, fname)
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        debut = f.read(200)
    if '<!DOCTYPE' in debut or '<html' in debut or 'Bad gateway' in debut:
        corrompus.append(fname)

# Exclure les Rest day (pas de GPX sur LFR, inutile de retry)
a_retelechager = [f for f in corrompus if 'Rest day' not in f]

print(f'Total corrompus : {len(corrompus)}')
print(f'Rest day exclus : {len(corrompus) - len(a_retelechager)}')
print(f'A re-télécharger : {len(a_retelechager)}')
for f in sorted(a_retelechager):
    print(f'  {f}')


Total corrompus : 142
Rest day exclus : 55
A re-télécharger : 87
  2018 Africa Cup - ITT.gpx
  2018 Africa Cup - TTT.gpx
  2018 Africa Cup.gpx
  2018 Burmese Road National Championships - ITT.gpx
  2018 Burmese Road National Championships.gpx
  2018 Qatar Road National Championships.gpx
  2018 Tour of Fuzhou Stage 4.gpx
  2019 Belizean Road National Championships.gpx
  2019 COPACI Campeonato Panamericano de Ruta Junior - ITT.gpx
  2019 COPACI Campeonato Panamericano de Ruta Junior.gpx
  2019 Chinese Road National Championships - TTT.gpx
  2019 Chinese Road National Championships.gpx
  2019 Cuban Road National Championships.gpx
  2019 Cyprus Road National Championships - ITT.gpx
  2019 Cyprus Road National Championships.gpx
  2019 Dominican Road National Championships.gpx
  2019 Georgian Road National Championships.gpx
  2019 Hongkonger Road National Championships - ITT.gpx
  2019 Jamaican Road National Championships - ITT.gpx
  2019 Kazakhstani Road National Championships.gpx
  2019 Ko

In [9]:
# ── 2. Retrouver les race_id depuis races.csv ────────────────────────────────
df_races = pd.read_csv(RACES_CSV)
race_id_map = {str(row['name']): str(row['id']) for _, row in df_races.iterrows()}

matches = {}
introuvables = []
for fname in a_retelechager:
    race_name = fname.replace('.gpx', '')
    if race_name in race_id_map:
        matches[fname] = race_id_map[race_name]
    else:
        # Correspondance partielle
        found = False
        for name, rid in race_id_map.items():
            if race_name.lower() in name.lower() or name.lower() in race_name.lower():
                matches[fname] = rid
                found = True
                break
        if not found:
            introuvables.append(fname)

print(f'Race ID trouvés : {len(matches)}/{len(a_retelechager)}')
if introuvables:
    print(f'Race ID introuvables ({len(introuvables)}) :')
    for f in introuvables:
        print(f'  {f}')


Race ID trouvés : 86/87
Race ID introuvables (1) :
  2023 Ocean Cup China PingTan International Road Cycling Race.gpx


In [10]:
# ── 3. Fonctions scraping (identiques au scraper original) ───────────────────

def get_gpx(race_id, driver):
    url = f'https://www.la-flamme-rouge.eu/maps/viewtrack/gpx/{race_id}'
    cookies = {c['name']: c['value'] for c in driver.get_cookies()}
    headers = {'User-Agent': driver.execute_script('return navigator.userAgent')}
    r = requests.get(url, cookies=cookies, headers=headers, allow_redirects=True, timeout=30)
    return r.content

def create_driver():
    options = uc.ChromeOptions()
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--window-size=1920,1080')
    return uc.Chrome(options=options, headless=False, version_main=148)

def login(driver, username, password):
    driver.get('https://www.la-flamme-rouge.eu')
    print('Attente Cloudflare...')
    time.sleep(5)
    if 'Just a moment' in driver.title:
        time.sleep(10)
    print('Cloudflare passé !')
    driver.get('https://www.la-flamme-rouge.eu/ucp.php?mode=login')
    WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.NAME, 'username')))
    for char in username:
        driver.find_element(By.NAME, 'username').send_keys(char)
        time.sleep(0.05)
    for char in password:
        driver.find_element(By.NAME, 'password').send_keys(char)
        time.sleep(0.05)
    driver.execute_script('arguments[0].click();', driver.find_element(By.NAME, 'login'))
    time.sleep(3)
    if 'incorrect' in driver.page_source.lower():
        print('Connexion échouée')
        return False
    print('Connexion réussie !')
    return True

print('Fonctions chargées.')


Fonctions chargées.


In [11]:
# ── 4. Connexion ─────────────────────────────────────────────────────────────
driver = create_driver()
login(driver, 'DeletangArthur', 'Marseille.37@')


Attente Cloudflare...
Cloudflare passé !
Connexion réussie !


True

In [12]:
# ── 5. Re-téléchargement — écrase le fichier corrompu ────────────────────────
succes = []
echecs = []

for fname, race_id in tqdm(matches.items()):
    output_path = os.path.join(GPX_DIR, fname)
    try:
        gpx_content = get_gpx(race_id, driver)
        if b'Just a moment' in gpx_content:
            print(f'  Cloudflare bloqué : {fname}')
            echecs.append(fname)
            continue
        if b'<trkpt' not in gpx_content and b'<wpt' not in gpx_content:
            print(f'  Contenu invalide : {fname}')
            echecs.append(fname)
            continue
        # Ecraser le fichier corrompu
        with open(output_path, 'wb') as f:
            f.write(gpx_content)
        print(f'  OK : {fname}')
        succes.append(fname)
    except Exception as e:
        print(f'  Erreur {fname} : {e}')
        echecs.append(fname)
    time.sleep(1)

print(f'\nSuccès : {len(succes)}')
print(f'Échecs  : {len(echecs)}')
if echecs:
    print('Fichiers en échec :')
    for f in echecs:
        print(f'  {f}')


  1%|          | 1/86 [00:01<01:26,  1.02s/it]

  Contenu invalide : 2022 National Road Championships - United Arab Emirates - ITT.gpx


  2%|▏         | 2/86 [00:01<01:13,  1.15it/s]

  Contenu invalide : 2020 Cyprus Road National Championships - ITT.gpx


  3%|▎         | 3/86 [00:02<01:07,  1.23it/s]

...

  Contenu invalide : 2020 Kosovian Road National Championships - ITT.gpx


 99%|█████████▉| 85/86 [01:50<00:01,  1.76s/it]

  Contenu invalide : 2019 Belizean Road National Championships.gpx
  OK : 2025 Tour de Romandie Feminin Stage 2.gpx


100%|██████████| 86/86 [01:52<00:00,  1.31s/it]


Succès : 18
Échecs  : 68
Fichiers en échec :
  2022 National Road Championships - United Arab Emirates - ITT.gpx
  2020 Cyprus Road National Championships - ITT.gpx
  2019 Hongkonger Road National Championships - ITT.gpx
  2019 Georgian Road National Championships.gpx
  2019 Marocaine Road National Championships - ITT.gpx
  2019 Mauritian Road National Championships.gpx
  2025 Tour du Rwanda.gpx
  2025 Vuelta Junior a la Ribera del Duero.gpx
  2019 Tunisian Road National Championships - ITT.gpx
  2018 Burmese Road National Championships.gpx
  2019 Kazakhstani Road National Championships.gpx
  2019 UAE Road Cycling Championships.gpx
  2019 Chinese Road National Championships - TTT.gpx
  2018 Qatar Road National Championships.gpx
  2018 Tour of Fuzhou Stage 4.gpx
  2020 Cuban Road National Championships.gpx
  2019 Saint Vincentian Road National Championships - ITT.gpx
  2021 61th Craft Ster van Zwolle.gpx
  2021 Ronde van Limburg.gpx
  2019 Virgin Islands Road National Championships - I

In [13]:
# ── 6. Vérification finale ───────────────────────────────────────────────────
driver.quit()

encore_corrompus = []
for fname in a_retelechager:
    path = os.path.join(GPX_DIR, fname)
    if not os.path.exists(path):
        encore_corrompus.append(f'ABSENT : {fname}')
        continue
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        debut = f.read(200)
    if '<!DOCTYPE' in debut or 'Bad gateway' in debut:
        encore_corrompus.append(f'TOUJOURS CORROMPU : {fname}')

print(f'Récupérés avec succès : {len(a_retelechager) - len(encore_corrompus)}/{len(a_retelechager)}')
if encore_corrompus:
    print('Encore problématiques :')
    for f in encore_corrompus:
        print(f'  {f}')


Récupérés avec succès : 18/87
Encore problématiques :
  TOUJOURS CORROMPU : 2022 National Road Championships - United Arab Emirates - ITT.gpx
  TOUJOURS CORROMPU : 2020 Cyprus Road National Championships - ITT.gpx
  TOUJOURS CORROMPU : 2019 Hongkonger Road National Championships - ITT.gpx
  TOUJOURS CORROMPU : 2019 Georgian Road National Championships.gpx
  TOUJOURS CORROMPU : 2019 Marocaine Road National Championships - ITT.gpx
  TOUJOURS CORROMPU : 2019 Mauritian Road National Championships.gpx
  TOUJOURS CORROMPU : 2025 Tour du Rwanda.gpx
  TOUJOURS CORROMPU : 2023 Ocean Cup China PingTan International Road Cycling Race.gpx
  TOUJOURS CORROMPU : 2025 Vuelta Junior a la Ribera del Duero.gpx
  TOUJOURS CORROMPU : 2019 Tunisian Road National Championships - ITT.gpx
  TOUJOURS CORROMPU : 2018 Burmese Road National Championships.gpx
  TOUJOURS CORROMPU : 2019 Kazakhstani Road National Championships.gpx
  TOUJOURS CORROMPU : 2019 UAE Road Cycling Championships.gpx
  TOUJOURS CORROMPU : 20